In [40]:
!pip install pandas numpy yfinance ta matplotlib

In [41]:
# =========================
# 1. IMPORTS
# =========================

import yfinance as yf
import pandas as pd
import numpy as np
import ta
import matplotlib.pyplot as plt

In [42]:
# =========================
# 2. INDICATORS
# =========================

def add_emas(df):
    df = df.copy()
    df["EMA9"] = df["Close"].ewm(span=9).mean()
    df["EMA20"] = df["Close"].ewm(span=20).mean()
    df["EMA50"] = df["Close"].ewm(span=50).mean()
    return df


def add_atr(df, window=14):
    df = df.copy()
    df["ATR"] = ta.volatility.average_true_range(
        high=df["High"],
        low=df["Low"],
        close=df["Close"],
        window=window
    )
    return df

In [14]:
# =========================
# 6. FILTERS AT A HISTORICAL TIME
# =========================

def check_filters_at_time(daily, hourly, current_time):
    current_time_clean = pd.Timestamp(current_time).tz_localize(None)
    current_date = current_time_clean.normalize()

    daily_clean = daily.copy()
    hourly_clean = hourly.copy()

    daily_clean.index = daily_clean.index.tz_localize(None)
    hourly_clean.index = hourly_clean.index.tz_localize(None)

    daily_past = daily_clean.loc[:current_date].iloc[:-1].copy()
    hourly_past = hourly_clean.loc[:current_time_clean].copy()

    if len(daily_past) < 100 or len(hourly_past) < 50:
        return {
            "monthly": False,
            "weekly": False,
            "daily": False,
            "hourly": False,
            "100D": False,
            "overall": False
        }

    weekly, monthly = build_weekly_monthly(daily_past)

    if len(monthly) < 12 or len(weekly) < 20:
        return {
            "monthly": False,
            "weekly": False,
            "daily": False,
            "hourly": False,
            "100D": False,
            "overall": False
        }

    last_month = monthly.iloc[-1]
    last_week = weekly.iloc[-1]
    last_day = daily_past.iloc[-1]
    recent_20_hours = hourly_past.tail(20)
    recent_15_hours = hourly_past.tail(15)

    monthly_pass = (
        last_month["Close"] > last_month["EMA9"]
        and last_month["Close"] > last_month["EMA20"]
        and last_month["Close"] > last_month["EMA50"]
    )

    weekly_pass = (
        last_week["Close"] > last_week["EMA9"]
        and last_week["Close"] > last_week["EMA20"]
        and last_week["Close"] > last_week["EMA50"]
    )

    daily_pass = (
        last_day["Close"] > last_day["EMA20"]
        and last_day["Close"] > last_day["EMA50"]
    )

    closed_below_50_recently = (
        recent_20_hours["Close"] < recent_20_hours["EMA50"]
    ).any()

    crossed_above_20_recently = (
        (recent_15_hours["Close"] > recent_15_hours["EMA20"])
        & (recent_15_hours["Close"].shift(1) <= recent_15_hours["EMA20"].shift(1))
    ).any()

    hundred_day_pass = hundred_day_high_recent(daily_past)

    hourly_pass = closed_below_50_recently and crossed_above_20_recently

    overall = (
        monthly_pass
        and weekly_pass
        and daily_pass
        and hourly_pass
        and hundred_day_pass
    )

    return {
        "monthly": monthly_pass,
        "weekly": weekly_pass,
        "daily": daily_pass,
        "hourly": hourly_pass,
        "100D": hundred_day_pass,
        "overall": overall
    }

In [16]:
def backtest_stats(trades):
    if len(trades) == 0:
        return {
            "Trades": 0,
            "Wins": 0,
            "Losses": 0,
            "Breakevens": 0,
            "Win Rate": None,
            "Average R": None,
            "Total R": 0,
            "Best R": None,
            "Worst R": None
        }

    wins = trades[trades["R Result"] > 0]
    losses = trades[trades["R Result"] < 0]
    breakevens = trades[trades["R Result"] == 0]

    return {
        "Trades": len(trades),
        "Wins": len(wins),
        "Losses": len(losses),
        "Breakevens": len(breakevens),
        "Win Rate": len(wins) / len(trades),
        "Average R": trades["R Result"].mean(),
        "Total R": trades["R Result"].sum(),
        "Best R": trades["R Result"].max(),
        "Worst R": trades["R Result"].min()
    }

In [66]:
BULLISH_REGIMES = ["Accumulation", "Reaccumulation"]


def prepare_regime_data(daily):
    daily_s, weekly_s, monthly_s = build_structure_timeframes(daily)

    daily_s = classify_structure_regime(daily_s)
    weekly_s = classify_structure_regime(weekly_s)
    monthly_s = classify_structure_regime(monthly_s)

    return daily_s, weekly_s, monthly_s


def get_regime_at_time(structure_df, current_time):
    current_date = pd.Timestamp(current_time).tz_localize(None).normalize()

    past = structure_df.loc[:current_date].iloc[:-1]

    if len(past) == 0:
        return "Neutral"

    regimes = past["Structure Regime"].dropna()

    if len(regimes) == 0:
        return "Neutral"

    return regimes.iloc[-1]


def check_precomputed_regimes(daily_s, weekly_s, monthly_s, current_time):
    daily_regime = get_regime_at_time(daily_s, current_time)
    weekly_regime = get_regime_at_time(weekly_s, current_time)
    monthly_regime = get_regime_at_time(monthly_s, current_time)

    overall = (
        daily_regime in BULLISH_REGIMES
        and weekly_regime in BULLISH_REGIMES
        and monthly_regime in BULLISH_REGIMES
    )

    return {
        "daily": daily_regime,
        "weekly": weekly_regime,
        "monthly": monthly_regime,
        "overall": overall
    }

In [67]:
def prepare_weekly_data(daily):
    weekly = daily.resample("W").agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
        "Volume": "sum"
    }).dropna()

    weekly = add_emas(weekly)
    weekly["Recent 10W High"] = weekly["High"].rolling(10).max()

    return weekly

In [52]:
# =========================
# REGIME FILTER AT HISTORICAL TIME
# =========================

BULLISH_REGIMES = ["Accumulation", "Reaccumulation"]


def get_latest_regime(structure_df):
    latest = structure_df["Structure Regime"].dropna()

    if len(latest) == 0:
        return "Neutral"

    return latest.iloc[-1]


def check_regime_filters_at_time(daily, current_time):
    current_date = pd.Timestamp(current_time).tz_localize(None).normalize()

    daily_clean = daily.copy()
    daily_clean.index = daily_clean.index.tz_localize(None)

    # only use completed daily candles before current signal day
    daily_past = daily_clean.loc[:current_date].iloc[:-1].copy()

    if len(daily_past) < 300:
        return {
            "daily": "Neutral",
            "weekly": "Neutral",
            "monthly": "Neutral",
            "overall": False
        }

    daily_s, weekly_s, monthly_s = build_structure_timeframes(daily_past)

    daily_s = classify_structure_regime(daily_s)
    weekly_s = classify_structure_regime(weekly_s)
    monthly_s = classify_structure_regime(monthly_s)

    daily_regime = get_latest_regime(daily_s)
    weekly_regime = get_latest_regime(weekly_s)
    monthly_regime = get_latest_regime(monthly_s)

    overall = (
        daily_regime in BULLISH_REGIMES
        and weekly_regime in BULLISH_REGIMES
        and monthly_regime in BULLISH_REGIMES
    )

    return {
        "daily": daily_regime,
        "weekly": weekly_regime,
        "monthly": monthly_regime,
        "overall": overall
    }

In [44]:
def build_shifted_timeframes(daily):
    weekly = daily.resample("W").agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
        "Volume": "sum"
    }).dropna()

    monthly = daily.resample("ME").agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
        "Volume": "sum"
    }).dropna()

    six_month = daily.resample("6ME").agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
        "Volume": "sum"
    }).dropna()

    weekly = add_emas(weekly)
    monthly = add_emas(monthly)
    six_month = add_emas(six_month)

    return weekly, monthly, six_month

In [45]:
def download_daily_data(symbol):
    daily = yf.download(symbol, period="10y", interval="1d", auto_adjust=False, progress=False)

    if daily.empty:
        raise ValueError(f"No data for {symbol}")

    daily = clean_columns(daily)

    if len(daily) < 500:
        raise ValueError(f"Not enough daily data for {symbol}")

    daily = add_emas(daily)
    daily = add_atr(daily)

    return daily

In [24]:
def check_shifted_filters_at_time(daily, current_time):
    current_date = pd.Timestamp(current_time).tz_localize(None).normalize()

    daily_clean = daily.copy()
    daily_clean.index = daily_clean.index.tz_localize(None)

    # use only completed daily candles before current signal day
    daily_past = daily_clean.loc[:current_date].iloc[:-1].copy()

    if len(daily_past) < 300:
        return {
            "six_month": False,
            "monthly": False,
            "weekly": False,
            "100W": False,
            "overall": False
        }

    weekly, monthly, six_month = build_shifted_timeframes(daily_past)

    if len(weekly) < 50 or len(monthly) < 20 or len(six_month) < 4:
        return {
            "six_month": False,
            "monthly": False,
            "weekly": False,
            "100W": False,
            "overall": False
        }

    last_6m = six_month.iloc[-1]
    last_month = monthly.iloc[-1]
    last_week = weekly.iloc[-1]

    six_month_pass = (
        last_6m["Close"] > last_6m["EMA9"]
        and last_6m["Close"] > last_6m["EMA20"]
    )

    monthly_pass = (
        last_month["Close"] > last_month["EMA9"]
        and last_month["Close"] > last_month["EMA20"]
        and last_month["Close"] > last_month["EMA50"]
    )

    weekly_pass = (
        last_week["Close"] > last_week["EMA20"]
        and last_week["Close"] > last_week["EMA50"]
    )

    hundred_week_pass = hundred_week_high_recent(daily_past)

    overall = (
        six_month_pass
        and monthly_pass
        and weekly_pass
        and hundred_week_pass
    )

    return {
        "six_month": six_month_pass,
        "monthly": monthly_pass,
        "weekly": weekly_pass,
        "100W": hundred_week_pass,
        "overall": overall
    }

In [64]:
def backtest_daily_strategy(symbol):
    daily = download_daily_data(symbol)

    daily_s, weekly_s, monthly_s = prepare_regime_data(daily)

    weekly_data = prepare_weekly_data(daily)

    trades = []

    locked = False
    lock_high = None

    order_active = False
    in_trade = False

    signal_high = None
    signal_time = None
    signal_atr = None
    order_created_index = None

    pullback_seen = False
    reclaim_seen = False

    entry_price = None
    stop_loss = None
    initial_risk = None
    entry_time = None
    moved_to_breakeven = False
    weekly_trail_active = False

    trade_taken_dates = set()

    for i in range(300, len(daily) - 10):
        current_time = daily.index[i]
        candle = daily.iloc[i]

        current_date = pd.Timestamp(current_time).tz_localize(None).normalize()
        trade_date = current_date.date()

        daily_past = daily.loc[:current_date].iloc[:-1]

        if len(daily_past) < 300:
            continue

        last_week = get_last_completed_week(weekly_data, current_time)

        if last_week is None:
            continue

        recent_10w_high = last_week["Recent 10W High"]

        # =========================
        # MANAGE OPEN TRADE
        # =========================
        if in_trade:
            

            # Move to breakeven after +1R
            if not moved_to_breakeven and candle["High"] >= entry_price + initial_risk:
                stop_loss = entry_price
                moved_to_breakeven = True

            # Weekly EMA9 trail only starts once weekly EMA9 is above breakeven/entry
            if moved_to_breakeven and last_week["EMA9"] > entry_price:
                weekly_trail_active = True

            # Exit when completed weekly candle closes below weekly EMA9
            if weekly_trail_active and last_week["Close"] < last_week["EMA9"]:
                exit_price = last_week["Close"]
                r_result = (exit_price - entry_price) / initial_risk

                trades.append({
                    "Ticker": symbol,
                    "Signal Time": signal_time,
                    "Entry Time": entry_time,
                    "Exit Time": last_week.name,
                    "Entry Price": entry_price,
                    "Exit Price": exit_price,
                    "Initial Stop": entry_price - initial_risk,
                    "Final Stop": last_week["EMA9"],
                    "R Result": r_result,
                    "Moved BE": moved_to_breakeven,
                    "Trail Active": weekly_trail_active,
                    "Exit Reason": "Weekly close below EMA9"
                })

                in_trade = False
                locked = True
                lock_high = recent_10w_high

                pullback_seen = False
                reclaim_seen = False
                weekly_trail_active = False
                continue

            # Normal stop loss before weekly trail exits
            if candle["Low"] <= stop_loss:
                exit_price = stop_loss
                r_result = (exit_price - entry_price) / initial_risk

                trades.append({
                    "Ticker": symbol,
                    "Signal Time": signal_time,
                    "Entry Time": entry_time,
                    "Exit Time": current_time,
                    "Entry Price": entry_price,
                    "Exit Price": exit_price,
                    "Initial Stop": entry_price - initial_risk,
                    "Final Stop": stop_loss,
                    "R Result": r_result,
                    "Moved BE": moved_to_breakeven,
                    "Trail Active": weekly_trail_active,
                    "Exit Reason": "Stop loss hit",
                    "Daily Regime": signal_daily_regime,
                    "Weekly Regime": signal_weekly_regime,
                    "Monthly Regime": signal_monthly_regime,
                })

                in_trade = False
                locked = True
                lock_high = recent_10w_high

                pullback_seen = False
                reclaim_seen = False
                weekly_trail_active = False
                continue

            continue

        # =========================
        # RESET LOCK
        # =========================
        if locked:
            if candle["High"] > lock_high:
                locked = False
                lock_high = None
                pullback_seen = False
                reclaim_seen = False
                order_active = False
            else:
                continue

        # =========================
        # MANAGE ACTIVE DAILY BUY STOP
        # =========================
        if order_active:
            candles_waited = i - order_created_index

            # Entry check first
            if candle["High"] > signal_high:
                entry_price = signal_high
                entry_time = current_time
                initial_risk = 1.5 * signal_atr
                stop_loss = entry_price - initial_risk
                moved_to_breakeven = False
                weekly_trail_active = False

                in_trade = True
                order_active = False
                trade_taken_dates.add(trade_date)
                continue

            # Cancel if drops 2 ATR below setup high
            if candle["Low"] <= signal_high - (2 * signal_atr):
                order_active = False
                signal_high = None
                signal_time = None
                signal_atr = None
                order_created_index = None
                pullback_seen = False
                reclaim_seen = False
                continue

            # Expires after 8 daily candles
            if candles_waited > 8:
                order_active = False
                signal_high = None
                signal_time = None
                signal_atr = None
                order_created_index = None
                pullback_seen = False
                reclaim_seen = False
                continue

            continue


        # =========================
        # DAILY SETUP
        # old hourly logic now on daily candles
        # =========================

        if not pullback_seen:
            if candle["Close"] < candle["EMA50"]:
                pullback_seen = True
            continue

        if pullback_seen and not reclaim_seen:
            if candle["Close"] > candle["EMA20"]:
                reclaim_seen = True
            continue

        if reclaim_seen:
            next_candle = daily.iloc[i + 1]

            # Max 1 trade per day
            if trade_date in trade_taken_dates:
                continue

            # Signal candle must close above EMA20
            if candle["Close"] <= candle["EMA20"]:
                continue

            # Signal candle must be bullish
            if candle["Close"] <= candle["Open"]:
                continue

            # Next candle must not break this candle's high
            if next_candle["High"] >= candle["High"]:
                continue

            # Do not create setup above/breaking recent 10-week high
            if candle["High"] >= recent_10w_high or candle["Open"] >= recent_10w_high:
                locked = True
                lock_high = recent_10w_high
                pullback_seen = False
                reclaim_seen = False
                continue

            filters = check_precomputed_regimes(
                daily_s,
                weekly_s,
                monthly_s,
                current_time
            )

            if not filters["overall"]:
                pullback_seen = False
                reclaim_seen = False
                continue

            signal_high = candle["High"]
            signal_time = current_time
            signal_atr = candle["ATR"]
            order_created_index = i
            order_active = True

            signal_daily_regime = filters["daily"]
            signal_weekly_regime = filters["weekly"]
            signal_monthly_regime = filters["monthly"]

            continue

    return pd.DataFrame(trades)

In [59]:
signal_daily_regime = None
signal_weekly_regime = None
signal_monthly_regime = None

In [26]:
def hundred_week_high_recent(daily_past, lookback_weeks=100, recent_weeks=8):
    weekly = daily_past.resample("W").agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
        "Volume": "sum"
    }).dropna()

    if len(weekly) < lookback_weeks:
        return False

    current_100_week_high = weekly["High"].rolling(lookback_weeks).max().iloc[-1]
    recent_high = weekly["High"].tail(recent_weeks).max()

    return recent_high >= current_100_week_high

In [29]:
# =========================
# CONFIRMATION-BASED MARKET STRUCTURE
# =========================

def add_confirmed_structure(df):
    df = df.copy()

    df["Bull Break"] = False
    df["Bear Break"] = False
    df["Bull Count"] = 0
    df["Bear Count"] = 0

    df["Candidate High"] = np.nan
    df["Candidate Low"] = np.nan
    df["Confirmed High"] = np.nan
    df["Confirmed Low"] = np.nan

    candidate_high = df["High"].iloc[0]
    candidate_low = df["Low"].iloc[0]

    confirmed_high = None
    confirmed_low = None

    bull_count = 0
    bear_count = 0

    state = "looking_for_high"

    for i in range(1, len(df)):
        candle = df.iloc[i]
        prev = df.iloc[i - 1]

        bearish_confirm = candle["Close"] < prev["Low"]
        bullish_confirm = candle["Close"] > prev["High"]

        # Track candidates
        candidate_high = max(candidate_high, candle["High"])
        candidate_low = min(candidate_low, candle["Low"])

        # Confirm a structure high after bearish close
        if bearish_confirm:
            confirmed_high = candidate_high
            candidate_low = candle["Low"]
            state = "waiting_for_bull_break"

        # Confirm a structure low after bullish close
        if bullish_confirm:
            confirmed_low = candidate_low
            candidate_high = candle["High"]
            state = "waiting_for_bear_break"

        # Bullish break of confirmed structure high
        if confirmed_high is not None and candle["Close"] > confirmed_high:
            bull_count += 1
            bear_count = 0

            df.iloc[i, df.columns.get_loc("Bull Break")] = True
            df.iloc[i, df.columns.get_loc("Bull Count")] = bull_count

            confirmed_low = candidate_low
            candidate_high = candle["High"]
            candidate_low = candle["Low"]
            confirmed_high = None

        # Bearish break of confirmed structure low
        if confirmed_low is not None and candle["Close"] < confirmed_low:
            bear_count += 1
            bull_count = 0

            df.iloc[i, df.columns.get_loc("Bear Break")] = True
            df.iloc[i, df.columns.get_loc("Bear Count")] = bear_count

            confirmed_high = candidate_high
            candidate_high = candle["High"]
            candidate_low = candle["Low"]
            confirmed_low = None

        df.iloc[i, df.columns.get_loc("Candidate High")] = candidate_high
        df.iloc[i, df.columns.get_loc("Candidate Low")] = candidate_low
        df.iloc[i, df.columns.get_loc("Confirmed High")] = confirmed_high
        df.iloc[i, df.columns.get_loc("Confirmed Low")] = confirmed_low

    return df

In [34]:
daily = download_daily_data("NOK")
structure = add_confirmed_structure(daily)

structure[
    structure["Bull Break"] | structure["Bear Break"]
][[
    "Open", "High", "Low", "Close",
    "Bull Break", "Bear Break",
    "Bull Count", "Bear Count",
    "Confirmed High", "Confirmed Low"
]].tail(50)

Price,Open,High,Low,Close,Bull Break,Bear Break,Bull Count,Bear Count,Confirmed High,Confirmed Low
Date,,,,,,,,,,
2024-08-06,3.720000,3.740000,3.700000,3.71,False,True,0,1,4.020000,NaN
2024-08-15,4.030000,4.100000,4.020000,4.07,True,False,1,0,NaN,3.70
2024-10-09,4.410000,4.470000,4.410000,4.47,True,False,2,0,NaN,4.33
2024-10-18,4.490000,4.780000,4.490000,4.75,True,False,3,0,NaN,4.14
2024-10-28,4.830000,4.950000,4.830000,4.95,True,False,4,0,NaN,4.66
2024-11-06,4.600000,4.620000,4.580000,4.58,False,True,0,1,4.950000,NaN
2024-11-19,4.410000,4.450000,3.910000,4.15,False,True,0,2,4.530000,NaN
2024-12-05,4.250000,4.340000,4.240000,4.31,True,False,1,0,NaN,4.16
2025-01-06,4.500000,4.580000,4.490000,4.56,True,False,2,0,NaN,4.39


In [63]:
def prepare_weekly_data(daily):
    weekly = daily.resample("W").agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
        "Volume": "sum"
    }).dropna()

    weekly = add_emas(weekly)
    weekly["Recent 10W High"] = weekly["High"].rolling(10).max()

    return weekly


def get_last_completed_week(weekly, current_time):
    current_date = pd.Timestamp(current_time).tz_localize(None).normalize()
    past = weekly.loc[:current_date].iloc[:-1]

    if len(past) == 0:
        return None

    return past.iloc[-1]

In [46]:
def build_structure_timeframes(daily):
    weekly = daily.resample("W").agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
        "Volume": "sum"
    }).dropna()

    monthly = daily.resample("ME").agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
        "Volume": "sum"
    }).dropna()

    daily_structure = add_confirmed_structure(daily)
    weekly_structure = add_confirmed_structure(weekly)
    monthly_structure = add_confirmed_structure(monthly)

    return daily_structure, weekly_structure, monthly_structure

In [47]:
def classify_structure_regime(df, lookback_breaks=10):
    df = df.copy()

    df["Structure Regime"] = None

    break_history = []
    last_major_side = None  # "bull" or "bear"

    for i in range(len(df)):
        row = df.iloc[i]

        if row["Bull Break"]:
            break_history.append(("bull", row["Bull Count"]))

        if row["Bear Break"]:
            break_history.append(("bear", row["Bear Count"]))

        recent = break_history[-lookback_breaks:]

        if len(recent) == 0:
            df.iloc[i, df.columns.get_loc("Structure Regime")] = "Neutral"
            continue

        current_side, current_count = recent[-1]

        # update major side once one side proves itself
        if current_side == "bull" and current_count >= 3:
            last_major_side = "bull"

        if current_side == "bear" and current_count >= 3:
            last_major_side = "bear"

        # classification
        if current_side == "bull":
            if last_major_side == "bear" and current_count < 3:
                regime = "Redistribution"
            else:
                regime = "Accumulation"

        elif current_side == "bear":
            if last_major_side == "bull" and current_count <= 2:
                regime = "Reaccumulation"
            else:
                regime = "Distribution"

        df.iloc[i, df.columns.get_loc("Structure Regime")] = regime

    return df

In [36]:
daily = download_daily_data("AAPL")

daily_structure = add_confirmed_structure(daily)
daily_structure = classify_structure_regime(daily_structure)

daily_structure[
    daily_structure["Bull Break"] | daily_structure["Bear Break"]
][[
    "Open", "High", "Low", "Close",
    "Bull Break", "Bear Break",
    "Bull Count", "Bear Count",
    "Structure Regime"
]].tail(50)

Price,Open,High,Low,Close,Bull Break,Bear Break,Bull Count,Bear Count,Structure Regime
Date,,,,,,,,,
2024-07-01,212.089996,217.509995,211.919998,216.750000,True,False,2,0,Accumulation
2024-07-15,236.479996,237.229996,233.089996,234.399994,True,False,3,0,Accumulation
2024-07-18,230.279999,230.440002,222.270004,224.179993,False,True,0,1,Reaccumulation
2024-08-05,199.089996,213.500000,196.000000,209.270004,False,True,0,2,Reaccumulation
2024-08-16,223.919998,226.830002,223.649994,226.050003,True,False,1,0,Accumulation
2024-08-29,230.100006,232.919998,228.880005,229.789993,True,False,2,0,Accumulation
2024-09-03,228.550003,229.000000,221.169998,222.770004,False,True,0,1,Reaccumulation
2024-09-16,216.539993,217.220001,213.919998,216.320007,False,True,0,2,Reaccumulation
2024-09-19,224.990005,229.820007,224.630005,228.869995,True,False,1,0,Accumulation


In [37]:
daily_s, weekly_s, monthly_s = build_structure_timeframes(daily)

daily_s = classify_structure_regime(daily_s)
weekly_s = classify_structure_regime(weekly_s)
monthly_s = classify_structure_regime(monthly_s)

In [38]:
weekly_s[
    weekly_s["Bull Break"] | weekly_s["Bear Break"]
][[
    "Open", "High", "Low", "Close",
    "Bull Break", "Bear Break",
    "Bull Count", "Bear Count",
    "Structure Regime"
]].tail(50)

Price,Open,High,Low,Close,Bull Break,Bear Break,Bull Count,Bear Count,Structure Regime
Date,,,,,,,,,
2016-09-18,25.662500,29.032499,25.632500,28.730000,True,False,1,0,Accumulation
2016-12-11,27.500000,28.674999,27.062500,28.487499,True,False,2,0,Accumulation
2017-05-07,36.275002,37.244999,36.067501,37.240002,True,False,3,0,Accumulation
2017-06-18,36.435001,36.875000,35.549999,35.567501,False,True,0,1,Reaccumulation
2017-08-06,37.474998,39.937500,37.032501,39.097500,True,False,1,0,Accumulation
2017-11-05,40.972500,43.564999,40.930000,43.125000,True,False,2,0,Accumulation
2018-01-21,44.474998,45.025002,43.767502,44.615002,True,False,3,0,Accumulation
2018-02-04,42.540001,42.540001,40.025002,40.125000,False,True,0,1,Reaccumulation
2018-02-18,39.625000,43.705002,39.377499,43.107498,True,False,1,0,Accumulation


In [39]:
monthly_s[
    monthly_s["Bull Break"] | monthly_s["Bear Break"]
][[
    "Open", "High", "Low", "Close",
    "Bull Break", "Bear Break",
    "Bull Count", "Bear Count",
    "Structure Regime"
]].tail(50)

Price,Open,High,Low,Close,Bull Break,Bear Break,Bull Count,Bear Count,Structure Regime
Date,,,,,,,,,
2017-01-31,28.950001,30.610001,28.690001,30.337500,True,False,1,0,Accumulation
2017-08-31,37.275002,41.130001,37.102501,41.000000,True,False,2,0,Accumulation
2019-09-30,51.607498,56.605000,51.055000,55.992500,True,False,3,0,Accumulation
2020-06-30,79.437500,93.095001,79.302498,91.199997,True,False,4,0,Accumulation
2021-07-31,136.600006,150.000000,135.759995,145.860001,True,False,5,0,Accumulation
2021-11-30,148.990005,165.699997,147.479996,165.300003,True,False,6,0,Accumulation
2022-06-30,149.899994,151.740005,129.039993,136.720001,False,True,0,1,Reaccumulation
2023-05-31,169.279999,179.350006,164.309998,177.250000,True,False,1,0,Accumulation
2024-06-30,192.899994,220.199997,192.149994,210.619995,True,False,2,0,Accumulation


In [48]:
# =========================
# BULLISH REGIME SCREENER
# =========================

BULLISH_REGIMES = ["Accumulation", "Reaccumulation"]


def get_latest_regime(structure_df):
    latest = structure_df["Structure Regime"].dropna()

    if len(latest) == 0:
        return "Neutral"

    return latest.iloc[-1]


def regime_screener(symbol):
    daily = download_daily_data(symbol)

    daily_s, weekly_s, monthly_s = build_structure_timeframes(daily)

    daily_s = classify_structure_regime(daily_s)
    weekly_s = classify_structure_regime(weekly_s)
    monthly_s = classify_structure_regime(monthly_s)

    daily_regime = get_latest_regime(daily_s)
    weekly_regime = get_latest_regime(weekly_s)
    monthly_regime = get_latest_regime(monthly_s)

    allowed = (
        daily_regime in BULLISH_REGIMES
        and weekly_regime in BULLISH_REGIMES
        and monthly_regime in BULLISH_REGIMES
    )

    return {
        "Ticker": symbol,
        "Daily Regime": daily_regime,
        "Weekly Regime": weekly_regime,
        "Monthly Regime": monthly_regime,
        "Trade Allowed": allowed
    }

In [49]:
regime_screener("AAPL")

{'Ticker': 'AAPL',
 'Daily Regime': 'Reaccumulation',
 'Weekly Regime': 'Accumulation',
 'Monthly Regime': 'Accumulation',
 'Trade Allowed': True}

In [50]:
watchlist = ["AAPL", "MSFT", "NVDA", "AMD", "META", "TSLA"]

rows = []

for ticker in watchlist:
    try:
        rows.append(regime_screener(ticker))
    except Exception as e:
        print(ticker, "skipped:", e)

screener = pd.DataFrame(rows)

screener[screener["Trade Allowed"] == True]

,Ticker,Daily Regime,Weekly Regime,Monthly Regime,Trade Allowed
0,AAPL,Reaccumulation,Accumulation,Accumulation,True
3,AMD,Accumulation,Accumulation,Accumulation,True
4,META,Reaccumulation,Reaccumulation,Reaccumulation,True


In [69]:
test_10 = [
    "AAPL", "MSFT", "NVDA", "AMZN", "META",
    "GOOGL", "AVGO", "JPM", "LLY", "XOM"
]

all_trades = []

for ticker in test_10:
    try:
        trades = backtest_daily_strategy(ticker)
        print(ticker, "trades:", len(trades))

        if len(trades) > 0:
            all_trades.append(trades)

    except Exception as e:
        print(ticker, "skipped:", e)

if len(all_trades) > 0:
    all_trades = pd.concat(all_trades, ignore_index=True)
else:
    all_trades = pd.DataFrame()

all_trades

AAPL trades: 8
MSFT trades: 5
NVDA trades: 4
AMZN trades: 4
META trades: 6
GOOGL trades: 7
AVGO trades: 3
JPM trades: 8
LLY trades: 9
XOM trades: 2


,Ticker,Signal Time,Entry Time,Exit Time,Entry Price,Exit Price,Initial Stop,Final Stop,R Result,Moved BE,Trail Active,Exit Reason,Daily Regime,Weekly Regime,Monthly Regime
0,AAPL,2018-02-16,2018-02-23,2018-03-19,43.705002,43.705002,42.065080,43.705002,0.000000,True,False,Stop loss hit,Accumulation,Reaccumulation,Accumulation
1,AAPL,2018-07-17,2018-07-19,2018-10-28,47.967499,54.075001,46.939166,54.602382,5.939226,True,True,Weekly close below EMA9,NaN,NaN,NaN
2,AAPL,2020-10-05,2020-10-09,2020-10-19,116.650002,116.650002,109.978044,116.650002,0.000000,True,False,Stop loss hit,Accumulation,Accumulation,Accumulation
3,AAPL,2021-04-09,2021-04-13,2021-05-04,133.039993,128.433050,128.433050,128.433050,-1.000000,False,False,Stop loss hit,Accumulation,Accumulation,Accumulation
4,AAPL,2021-10-20,2021-10-22,2022-01-23,149.750000,162.410004,145.686543,167.724793,3.115575,True,True,Weekly close below EMA9,NaN,NaN,NaN
5,AAPL,2023-11-02,2023-11-06,2024-01-07,177.779999,181.179993,172.799850,188.352711,0.682709,True,True,Weekly close below EMA9,NaN,NaN,NaN
6,AAPL,2024-08-16,2024-08-20,2024-09-04,226.830002,218.231165,218.231165,218.231165,-1.000000,False,False,Stop loss hit,Reaccumulation,Accumulation,Accumulation
7,AAPL,2024-11-19,2024-11-22,2025-01-13,230.160004,230.160004,224.152072,230.160004,0.000000,True,True,Stop loss hit,Reaccumulation,Accumulation,Accumulation
8,MSFT,2020-10-05,2020-10-08,2020-10-26,210.410004,210.410004,201.328402,210.410004,0.000000,True,True,Stop loss hit,Reaccumulation,Reaccumulation,Accumulation
9,MSFT,2021-03-11,2021-03-16,2021-03-18,239.169998,230.598151,230.598151,230.598151,-1.000000,False,False,Stop loss hit,Accumulation,Accumulation,Accumulation


In [70]:
backtest_stats(all_trades)

{'Trades': 56,
 'Wins': 15,
 'Losses': 21,
 'Breakevens': 20,
 'Win Rate': 0.26785714285714285,
 'Average R': np.float64(0.8690482902433282),
 'Total R': np.float64(48.66670425362638),
 'Best R': 10.512645502941705,
 'Worst R': -2.0592557733393693}